# Tutorial 14: EmbeddingStore and `adata.embpy`

This temporary tutorial shows the first MVP of the embpy semantic embedding layer.

It uses tiny synthetic data so you can run everything locally without downloading models or calling resolvers.

The key idea:

- **AnnData** remains the experiment container: expression, observations, variables, `.obsm`, `.varm`.
- **EmbeddingStore** is a reusable embedding universe: genes, proteins, molecules, cytokines, text, and typed relations.
- **`adata.embpy`** connects one experiment to those reusable embeddings, runs quick analyses, reuses `embpy.tl`/`embpy.pl`, and compiles perturbation/action embeddings for ML.

Generated embeddings still never go into `.X`.

## 1. Setup

We import only lightweight dependencies and force a non-interactive matplotlib backend so the notebook is safe to run in CI or from a terminal.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from anndata import AnnData

from embpy.io.result import EmbeddingProvenance, EmbeddingResult
from embpy.store import EmbeddingStore

np.set_printoptions(precision=3, suppress=True)

## 2. Create a tiny AnnData experiment

Think of this as a miniature perturbation experiment:

- six cells/observations
- three genes/features
- three perturbation conditions: `g1`, `g2`, and the combo `g1+g2`
- expression/count-like data remains in `.X`

In [ ]:
obs = pd.DataFrame(
    {
        "perturbation": ["g1", "g1", "g2", "g2", "g1+g2", "g1+g2"],
        "target_group": ["g1", "g1", "g2", "g2", "combo", "combo"],
        "phenotype_score": [1.0, 1.2, 3.0, 2.8, 2.0, 2.2],
        "cell_type": ["T", "T", "B", "B", "T", "B"],
    },
    index=[f"cell{i}" for i in range(6)],
)
var = pd.DataFrame(index=["g1", "g2", "g3"])
X = np.array(
    [
        [10, 2, 0],
        [11, 1, 0],
        [2, 10, 1],
        [3, 9, 1],
        [6, 6, 2],
        [5, 7, 2],
    ],
    dtype=np.float32,
)
adata = AnnData(X=X, obs=obs, var=var)
adata

## 3. Create canonical embedding results

In real usage these would come from `BioEmbedder.embed(...)`. Here we construct small `EmbeddingResult` objects directly so the tutorial stays fast.

The gene embedding result uses canonical gene IDs from `adata.var_names`, and the cell embedding matrix is aligned to `adata.obs_names`.

In [ ]:
gene_result = EmbeddingResult(
    matrix=np.array(
        [
            [1.0, 0.0],
            [0.0, 1.0],
            [0.25, 0.25],
        ],
        dtype=np.float32,
    ),
    entity_ids=("g1", "g2", "g3"),
    entity_type="gene",
    id_scheme="symbol",
    provenance=EmbeddingProvenance(model="toy_gene_model", pooling="mean"),
    aliases={"g1": {"display_name": "Gene 1"}, "g2": {"display_name": "Gene 2"}},
)

cell_embedding = np.array(
    [
        [1.0, 0.0],
        [0.9, 0.1],
        [0.0, 1.0],
        [0.1, 0.9],
        [0.5, 0.5],
        [0.45, 0.55],
    ],
    dtype=np.float32,
)

gene_result

## 4. Build an `EmbeddingStore`

`EmbeddingStore` is the external reusable embedding universe. It is useful for things that are too large or too reusable to stuff into every AnnData object: all genes, all proteins, huge molecule libraries, cytokines, pathways, text descriptions, and relation tables.

The store does not run model inference or canonicalization. It stores already-canonical embeddings and relations.

In [ ]:
store = EmbeddingStore()
store.add_result(gene_result, key="gene:toy_gene_model")
store.add_relation(
    "perturbation_targets_gene",
    pd.DataFrame(
        {
            "source_id": ["g1", "g2", "g1+g2", "g1+g2"],
            "target_id": ["g1", "g2", "g1", "g2"],
        }
    ),
    source_type="perturbation",
    target_type="gene",
)

store.describe()

In [ ]:
store.audit()

## 5. Write/read `.emstore`

The MVP `.emstore` format is a directory:

```text
example.emstore/
  manifest.json
  entities/*.parquet
  embeddings/*/matrix.npy
  embeddings/*/index.parquet
  relations/*.parquet
```

Matrices can be memory-mapped on read with `backed=True`, which matters for large molecule/cytokine/protein universes.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out = store.write(Path(tmp) / "toy.emstore")
    loaded = EmbeddingStore.read(out, backed=True)
    print(out)
    print(type(loaded.embedding("gene:toy_gene_model").matrix))
    display(loaded.describe())

## 6. Connect the store to AnnData with `adata.embpy`

`adata.embpy` keeps a semantic registry in `adata.uns["embpy"]`.

Aligned matrices go to AnnData-native slots:

- observation/cell embeddings -> `.obsm`
- gene/feature embeddings -> `.varm`
- large non-aligned reusable embeddings -> linked `EmbeddingStore`

`.X` is left untouched.

In [ ]:
X_before = adata.X.copy()

adata.embpy.register_store(store)
adata.embpy.register_embedding(
    "X_cells_toy",
    cell_embedding,
    entity_ids=adata.obs_names,
    entity_type="cell",
    id_scheme="obs_name",
)
adata.embpy.register_embedding("X_gene_toy", result=gene_result)

print("obsm:", list(adata.obsm.keys()))
print("varm:", list(adata.varm.keys()))
print("X unchanged:", np.array_equal(adata.X, X_before))
adata.embpy.list_embeddings()

## 7. Audit the registry

`audit()` checks the semantic registry, linked stores, and relation coverage. A clean table means no current issues were found.

In [ ]:
adata.embpy.describe()

In [ ]:
adata.embpy.audit()

## 8. Aggregate embeddings by biological groups

This wraps reusable `embpy.tl` logic. The accessor only resolves the registered embedding and the AnnData metadata column.

Here we compute perturbation-level centroids from cell-level embeddings.

In [ ]:
centroids = adata.embpy.aggregate("X_cells_toy", by="perturbation")
centroids

## 9. Nearest neighbors and similarity/correlation analyses

`adata.embpy.neighbors(...)` returns a tidy table. Under the hood it delegates to `embpy.tl.nearest_neighbors_table(...)`.

`correlate(...)` compares embedding geometry to another embedding space or to a phenotype column.

In [ ]:
adata.embpy.neighbors("X_cells_toy", query="cell0", k=3)

In [ ]:
adata.embpy.correlate("X_cells_toy", phenotype="phenotype_score")

In [ ]:
# Register a second toy embedding so we can compare spaces.
adata.embpy.register_embedding(
    "X_cells_toy_flipped",
    cell_embedding[:, ::-1],
    entity_ids=adata.obs_names,
    entity_type="cell",
    id_scheme="obs_name",
)
adata.embpy.compare_embeddings(["X_cells_toy", "X_cells_toy_flipped"])

## 10. Reuse existing plotting functions

These wrappers call existing `embpy.pl` functions. The goal is not to create a new plotting ecosystem; it is to make the correct plot easy to call from the semantic registry.

In [ ]:
fig = adata.embpy.plot_embedding("X_cells_toy", method="pca", color="perturbation")
plt.close(fig)
fig

In [ ]:
fig = adata.embpy.plot_similarity("X_cells_toy", label_col="perturbation", title="Toy cell embedding similarity")
plt.close(fig)
fig

In [ ]:
fig = adata.embpy.plot_model_comparison(["X_cells_toy", "X_cells_toy_flipped"], method="cosine_correlation")
plt.close(fig)
fig

In [ ]:
fig = adata.embpy.plot_diagnostics(["X_cells_toy", "X_cells_toy_flipped"], kind="norms")
plt.close(fig)
fig

## 11. Phenotypic activity

This wraps `embpy.tl.phenotypic_activity`, which computes replicate clustering/activity using chunked cosine similarity.

In a real screen this can be used to ask which perturbations produce consistent phenotypic embeddings.

In [ ]:
activity = adata.embpy.score_activity("X_cells_toy", perturbation_col="perturbation", chunk_size=3)
activity

## 12. Set up conditions and compile action embeddings

This is the perturbation-ML bridge.

`setup_conditions(...)` defines stable condition IDs. `compile_actions(...)` uses the linked store and relation table to turn perturbation labels into action vectors.

For example, `g1+g2` becomes the mean of the `g1` and `g2` gene embeddings.

In [ ]:
conditions = adata.embpy.setup_conditions(condition_key="perturbation", control_values=[])
conditions.head()

In [ ]:
action_table = adata.embpy.compile_actions(
    output_key="X_embpy_action",
    relation="perturbation_targets_gene",
    target_embedding="gene:toy_gene_model",
    perturbation_key="perturbation",
    aggregation="mean",
)

action_table

In [ ]:
print("compiled action embedding shape:", adata.obsm["X_embpy_action"].shape)
print("g1+g2 action vector:", action_table.loc["g1+g2"].to_numpy())

## 13. Leakage-aware splits

This MVP supports deterministic random splits by a biological grouping column. Splitting by `target_group`, for example, avoids mixing the same target group between train and test.

In [ ]:
splits = adata.embpy.make_splits(by="target_group", random_state=1)
for name, idx in splits.items():
    print(name, idx.tolist(), adata.obs.iloc[idx]["target_group"].unique().tolist())

## 14. Make a Torch dataset

The accessor can compile aligned state/action/target tensors for simple ML workflows.

For this MVP, the dataset is intentionally minimal. It is a bridge, not a replacement for the richer `world_model` dataloaders.

In [ ]:
try:
    dataset = adata.embpy.make_torch_dataset(split=splits["train"], action_key="X_embpy_action")
    sample = dataset[0]
    print(sample.keys())
    print("state", sample["state"].shape)
    print("action", sample["action"].shape)
    print("target", sample["target"].shape)
except ImportError as exc:
    print(exc)

## 15. Mental model

Use this split when designing new workflows:

```text
AnnData
  one experiment: cells, expression, obs/var metadata, aligned embeddings

EmbeddingStore
  reusable universe: genes, molecules, proteins, cytokines, pathways, text, relations

adata.embpy
  bridge: registry, audits, group-level summaries, plots, action compilation, ML datasets
```

If a function is general numerical analysis, it should live in `embpy.tl`.
If it is a visualization, it should live in `embpy.pl`.
If it is semantic orchestration across AnnData and stores, it belongs in `adata.embpy`.